# Consumer Credit Risk Analysis — Probability of Default Modeling

## Objective

Build and evaluate logistic regression models to estimate borrower-level probability of default.

This notebook covers:

- Loading the prepared modeling dataset
- Selecting model features
- Train/test split
- Feature scaling
- Logistic regression
- ROC-AUC and accuracy evaluation
- Model comparison
- Predicted probability analysis
- Risk-decile segmentation

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

## 1. Load Prepared Modeling Data

Load the cleaned modeling population created in the data preparation notebook. The target variable, `default_flag`, equals 1 for charged-off loans and 0 for fully paid loans.

In [2]:
model_df = pd.read_csv(
    "../data/processed/model_data.csv",
    low_memory=False
)

print(model_df.shape)
model_df[["fico", "dti", "annual_inc", "loan_amnt", "default_flag"]].head()

(1632364, 26)


,fico,dti,annual_inc,loan_amnt,default_flag
0,717.0,27.74,42000.0,12000.0,0
1,697.0,18.79,55000.0,10000.0,0
2,702.0,20.36,120000.0,8000.0,0
3,662.0,22.63,90000.0,12800.0,0
4,682.0,38.07,180000.0,15000.0,0


## 2. Baseline Model Features

The baseline logistic regression uses four borrower and loan characteristics:

- **FICO score** — borrower credit quality
- **DTI** — borrower debt burden relative to income
- **Annual income** — borrower repayment capacity
- **Loan amount** — size of the credit exposure

The target variable is `default_flag`, where 1 represents a charged-off loan and 0 represents a fully paid loan.

In [3]:
model_features = [
    "fico",
    "dti",
    "annual_inc",
    "loan_amnt"
]

model_data = model_df[
    model_features + ["default_flag"]
].dropna().copy()

print("Modeling rows:", len(model_data))
print("\nMissing values:")
print(model_data.isnull().sum())

print("\nDefault rate:")
print(model_data["default_flag"].mean())

Modeling rows: 1631256

Missing values:
fico            0
dti             0
annual_inc      0
loan_amnt       0
default_flag    0
dtype: int64

Default rate:
0.2004492244013202


## 3. Train/Test Split

The modeling dataset is divided into training and test samples.

- **80% training data** is used to estimate the logistic regression model.
- **20% test data** is held out to evaluate performance on unseen observations.
- Stratification preserves approximately the same default rate in both samples.

In [4]:
X = model_data[model_features]
y = model_data["default_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

Training rows: 1305004
Test rows: 326252

Training default rate: 0.20044919402545894
Test default rate: 0.20044934590439292


## 4. Feature Standardization

The predictor variables are standardized before fitting the logistic regression model. Standardization places variables with different units and magnitudes on a comparable scale.

The scaler is fitted only on the training data to prevent information from the test set from influencing model training.

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Training shape: (1305004, 4)
Test shape: (326252, 4)


## 5. Baseline Logistic Regression Model

A logistic regression model is trained using FICO, DTI, annual income, and loan amount to estimate each borrower's probability of default.

In [6]:
model_1 = LogisticRegression(max_iter=1000)

model_1.fit(X_train_scaled, y_train)

print("Model trained successfully.")

Model trained successfully.


## 6. Model Coefficients

The logistic regression coefficients show the direction and relative strength of the relationship between each standardized predictor and default risk.

- Positive coefficient → associated with higher default risk
- Negative coefficient → associated with lower default risk

In [7]:
coefficients = pd.DataFrame({
    "feature": model_features,
    "coefficient": model_1.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

coefficients

,feature,coefficient,abs_coefficient
2,annual_inc,-0.625715,0.625715
0,fico,-0.377415,0.377415
3,loan_amnt,0.312359,0.312359
1,dti,0.112495,0.112495


## 7. Model Evaluation

The baseline model is evaluated on the held-out test sample using ROC-AUC and accuracy.

ROC-AUC measures the model's ability to rank defaulted loans as riskier than non-defaulted loans, while accuracy measures the percentage of observations classified correctly at the default classification threshold.

In [8]:
y_prob = model_1.predict_proba(X_test_scaled)[:, 1]
y_pred = model_1.predict(X_test_scaled)

auc = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("ROC-AUC:", round(auc, 4))
print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(cm)

ROC-AUC: 0.6321
Accuracy: 0.7992

Confusion Matrix:
[[260658    197]
 [ 65303     94]]


## 8. Expanded Logistic Regression Model

The second model expands the baseline specification by adding two LendingClub risk-pricing variables:

- **Interest rate**
- **Credit grade**

Credit grades A through G are converted to an ordinal numeric scale from 1 through 7.

Model 2 is used to evaluate whether LendingClub's risk assessment and pricing variables provide additional predictive information beyond borrower characteristics alone.

In [9]:
grade_map = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

model_df["grade_num"] = model_df["grade"].map(grade_map)

model_2_features = [
    "fico",
    "dti",
    "annual_inc",
    "loan_amnt",
    "int_rate_num",
    "grade_num"
]

model_2_data = model_df[
    model_2_features + ["default_flag"]
].dropna().copy()

print("Model 2 rows:", len(model_2_data))
print("\nMissing values:")
print(model_2_data.isnull().sum())

print("\nGrade mapping check:")
print(
    model_df[["grade", "grade_num"]]
    .drop_duplicates()
    .sort_values("grade_num")
)

Model 2 rows: 1631256

Missing values:
fico            0
dti             0
annual_inc      0
loan_amnt       0
int_rate_num    0
grade_num       0
default_flag    0
dtype: int64

Grade mapping check:
   grade  grade_num
0      A          1
1      B          2
2      C          3
8      D          4
7      E          5
10     F          6
12     G          7


### Model 2 Train/Test Split and Standardization

The expanded model uses the same 80/20 train-test methodology as the baseline model. Features are standardized using parameters learned from the training sample only.

In [10]:
X2 = model_2_data[model_2_features]
y2 = model_2_data["default_flag"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.20,
    random_state=42,
    stratify=y2
)

scaler_2 = StandardScaler()

X2_train_scaled = scaler_2.fit_transform(X2_train)
X2_test_scaled = scaler_2.transform(X2_test)

print("Training rows:", len(X2_train))
print("Test rows:", len(X2_test))
print("Training default rate:", y2_train.mean())
print("Test default rate:", y2_test.mean())

Training rows: 1305004
Test rows: 326252
Training default rate: 0.20044919402545894
Test default rate: 0.20044934590439292


### Model 2 Training and Evaluation

The expanded logistic regression model is trained using borrower characteristics plus interest rate and credit grade. Performance is evaluated on the held-out test sample and compared with the baseline model.

In [11]:
model_2 = LogisticRegression(max_iter=1000)

model_2.fit(X2_train_scaled, y2_train)

# Generate predictions
y2_prob = model_2.predict_proba(X2_test_scaled)[:, 1]
y2_pred = model_2.predict(X2_test_scaled)

# Evaluate
auc_2 = roc_auc_score(y2_test, y2_prob)
accuracy_2 = accuracy_score(y2_test, y2_pred)
cm_2 = confusion_matrix(y2_test, y2_pred)

print("ROC-AUC:", round(auc_2, 4))
print("Accuracy:", round(accuracy_2, 4))

print("\nConfusion Matrix:")
print(cm_2)

ROC-AUC: 0.6917
Accuracy: 0.7996

Confusion Matrix:
[[257066   3789]
 [ 61585   3812]]


### Model 2 Coefficients

Standardized logistic regression coefficients are examined to understand the direction and relative strength of each predictor's relationship with default risk.

In [12]:
coefficients_2 = pd.DataFrame({
    "feature": model_2_features,
    "coefficient": model_2.coef_[0]
})

coefficients_2["abs_coefficient"] = coefficients_2["coefficient"].abs()

coefficients_2 = coefficients_2.sort_values(
    "abs_coefficient",
    ascending=False
)

coefficients_2

,feature,coefficient,abs_coefficient
2,annual_inc,-0.355709,0.355709
4,int_rate_num,0.338868,0.338868
5,grade_num,0.216982,0.216982
3,loan_amnt,0.181538,0.181538
0,fico,-0.146512,0.146512
1,dti,0.046371,0.046371


## 9. Risk Decile Analysis

Predicted probabilities from Model 2 are divided into ten equally sized risk groups.

Decile 1 represents the lowest predicted-risk borrowers, while Decile 10 represents the highest predicted-risk borrowers. Comparing predicted and actual default rates across deciles evaluates the model's ability to rank-order credit risk.

In [13]:
decile_df = pd.DataFrame({
    "actual_default": y2_test.values,
    "predicted_pd": y2_prob
})

decile_df["risk_decile"] = pd.qcut(
    decile_df["predicted_pd"],
    q=10,
    labels=range(1, 11)
)

decile_summary = (
    decile_df
    .groupby("risk_decile", observed=False)
    .agg(
        loans=("actual_default", "size"),
        defaults=("actual_default", "sum"),
        actual_default_rate=("actual_default", "mean"),
        avg_predicted_pd=("predicted_pd", "mean")
    )
)

decile_summary["actual_default_rate"] *= 100
decile_summary["avg_predicted_pd"] *= 100

decile_summary.round(2)

,loans,defaults,actual_default_rate,avg_predicted_pd
risk_decile,,,,
1,32626,1618,4.96,6.92
2,32625,2842,8.71,9.72
3,32625,3776,11.57,12.14
4,32625,4585,14.05,14.18
5,32625,5388,16.51,16.33
6,32625,6414,19.66,18.90
7,32625,7718,23.66,21.49
8,32625,8875,27.20,25.14
9,32625,10562,32.37,31.09


### Default Capture by Risk Decile

Default capture measures how effectively the model concentrates actual defaults within the highest predicted-risk segments. This provides a practical measure of how the model could support underwriting review, monitoring, or credit strategy.

In [14]:
# Calculate share of total defaults in each decile
decile_summary["default_capture_pct"] = (
    decile_summary["defaults"] /
    decile_summary["defaults"].sum()
) * 100

# Calculate cumulative capture starting with highest-risk decile
capture_table = decile_summary.sort_index(ascending=False).copy()

capture_table["cumulative_default_capture_pct"] = (
    capture_table["default_capture_pct"].cumsum()
)

capture_table[
    [
        "loans",
        "defaults",
        "actual_default_rate",
        "default_capture_pct",
        "cumulative_default_capture_pct"
    ]
].round(2)

,loans,defaults,actual_default_rate,default_capture_pct,cumulative_default_capture_pct
risk_decile,,,,,
10,32626,13619,41.74,20.83,20.83
9,32625,10562,32.37,16.15,36.98
8,32625,8875,27.20,13.57,50.55
7,32625,7718,23.66,11.80,62.35
6,32625,6414,19.66,9.81,72.16
5,32625,5388,16.51,8.24,80.40
4,32625,4585,14.05,7.01,87.41
3,32625,3776,11.57,5.77,93.18
2,32625,2842,8.71,4.35,97.53


## 10. Model Conclusions

The probability-of-default modeling produced several key findings:

- The baseline logistic regression model achieved a **ROC-AUC of 0.6321** using FICO, DTI, annual income, and loan amount.
- Adding LendingClub credit grade and interest rate improved ROC-AUC to **0.6917**, indicating materially better risk discrimination.
- Model 2 demonstrated strong risk ordering across deciles. Actual default rates increased from **5.0% in the lowest-risk decile to 41.7% in the highest-risk decile**.
- The highest-risk **10% of loans captured 20.8% of observed defaults**, while the highest-risk **30% captured 50.6% of observed defaults**.
- Accuracy remained near 80% for both models and was less informative because approximately 80% of the portfolio consisted of non-defaulted loans.
- The results demonstrate that predicted probability of default can be used to segment borrowers into materially different risk groups.

### Business Implication

Rather than relying on a single borrower characteristic or a simple default classification, lenders can use predicted probability of default to rank applicants and portfolio exposures by relative risk.

These risk rankings could support strategies such as enhanced underwriting review, credit-limit decisions, risk-based pricing, and portfolio monitoring. Any production credit decisioning framework would require additional validation, fairness testing, calibration, and governance before implementation.